In [77]:
model = 'Here is the completed code segment with all population data filled in:\n\n```problog\n{\n  "HASH": "23C2CC6E",\n  "Code": "pop(china, 8250).\npop(india, 5863).\npop(ussr, 2521).\npop(usa, 2119).\npop(indonesia, 1276).\npop(japan, 1097).\npop(brazil, 1042).\npop(bangladesh, 750).\npop(pakistan, 682).\npop(w_germany, 613).\npop(nigeria, 613).\npop(mexico, 581).\npop(uk, 559).\npop(italy, 554).\npop(france, 525).\npop(philippines, 415).\npop(thailand, 410).\npop(turkey, 383).\npop(egypt, 364).\npop(spain, 352).\npop(poland, 337).\npop(s_korea, 335).\npop(iran, 320).\npop(ethiopia, 272).\npop(argentina, 251)."\n}\n```'


In [85]:
import re
import json
import uuid
import hashlib
from problog.program import PrologString, Clause, AnnotatedDisjunction, Term
from typing import Literal, List, Union, Tuple, Dict, Any
from langgraph.graph import END, StateGraph
from state import BasicState, Mode
from config import paths
def _find_all_blocks(type:Literal["report","code","other"], text:str, ext_mark:str="") -> List[dict]:
    """
    find block with certain patterns in json form.
    """
    blocks:List[dict] = []
    
    if type == "other" or type == "report":
        pattern = r"```(?:report|[a-z]*)?\n(.*?)```"
    elif type == "code":
        pattern = r"```(?:problog|[a-z]*)?\n(.*?)```"
    else:
        raise ValueError("you must choose from ['report','code','other']")
    
    matches = re.findall(pattern, text, re.DOTALL)
    if not matches:
        pattern = r"```(?:json|[a-z]*)?\n(.*?)```"
        matches = re.findall(pattern, text, re.DOTALL)

    if matches:
        for match in matches:
            try:
                # 直接尝试解析 JSON
                match_json = json.loads(match)
            except json.JSONDecodeError:
                # 对于 ProbLog 代码，需要特殊处理转义
                # 保护 ProbLog 特殊操作符
                protected_match = match
                
                # 创建特殊操作符的映射
                special_patterns = {
                    r'\\\\=': '<<<BACKSLASH_BACKSLASH_EQUALS>>>',
                    r'\\\\\\\\=': '<<<QUAD_BACKSLASH_EQUALS>>>',  # 处理已经有四个反斜杠的情况
                    r'\\n'        : '<<<ESCAPED_NEWLINE>>>',      # 新增这一行
                }
                # 临时替换特殊操作符
                for pattern, placeholder in special_patterns.items():
                    protected_match = re.sub(pattern, placeholder, protected_match)
                
                # 修复其他无效的转义序列
                fixed_code = re.sub(r'\\(?!["\\/bfnrtu])', r'\\\\', protected_match)

                # 还原特殊操作符
                for pattern, placeholder in special_patterns.items():
                    fixed_code = fixed_code.replace(placeholder, pattern)
                
                try:
                    match_json = json.loads(fixed_code)
                except json.JSONDecodeError as e:
                    print(f"Failed to parse JSON: {e}")
                    print(f"Original match: {match}")
                    print(f"Fixed code: {fixed_code}")
                    continue

            # 处理不同类型的块
            if type == "other":
                try:
                    blocks.append({ext_mark:match_json})
                except ValueError:
                    blocks.append({ext_mark:None})

            elif type == "code":
                try:
                    blocks.append({match_json["HASH"]:match_json["Code"]})
                except (ValueError, KeyError):
                    blocks.append({match_json.get("HASH", "unknown"):f"could not parse the code block of {match_json}"})
            
            elif type == "report":
                try:
                    blocks.append({match_json["HASH"]:match_json})
                except (ValueError, KeyError):
                    blocks.append({match_json.get("HASH", "unknown"):f"could not parse the report block of {match_json}"})

        return blocks
    else:
        return None

In [84]:
import json
matches = _find_all_blocks("code",model)
print(matches)
# data 现在就是一个标准 dict，可以直接操作


fixed_code {
  "HASH": "23C2CC6E",
  "Code": "pop(china, 8250).
pop(india, 5863).
pop(ussr, 2521).
pop(usa, 2119).
pop(indonesia, 1276).
pop(japan, 1097).
pop(brazil, 1042).
pop(bangladesh, 750).
pop(pakistan, 682).
pop(w_germany, 613).
pop(nigeria, 613).
pop(mexico, 581).
pop(uk, 559).
pop(italy, 554).
pop(france, 525).
pop(philippines, 415).
pop(thailand, 410).
pop(turkey, 383).
pop(egypt, 364).
pop(spain, 352).
pop(poland, 337).
pop(s_korea, 335).
pop(iran, 320).
pop(ethiopia, 272).
pop(argentina, 251)."
}

Failed to parse JSON: Invalid control character at: line 3 column 29 (char 52)
Original match: {
  "HASH": "23C2CC6E",
  "Code": "pop(china, 8250).
pop(india, 5863).
pop(ussr, 2521).
pop(usa, 2119).
pop(indonesia, 1276).
pop(japan, 1097).
pop(brazil, 1042).
pop(bangladesh, 750).
pop(pakistan, 682).
pop(w_germany, 613).
pop(nigeria, 613).
pop(mexico, 581).
pop(uk, 559).
pop(italy, 554).
pop(france, 525).
pop(philippines, 415).
pop(thailand, 410).
pop(turkey, 383).
pop(egypt, 364).

In [65]:
for match in matches:
    print(match)
    res = json.loads(match)
    print(res)

In [6]:
def _find_all_blocks(type: Literal["report", "code", "other"], text: str, ext_mark: str = "") -> List[dict]:
    """
    修复版的解析函数，正确处理ProbLog代码中的转义序列
    """
    blocks: List[dict] = []
    
    # 根据类型选择正则表达式
    if type == "other" or type == "report":
        pattern = r"```(?:report|[a-z]*)?\n(.*?)```"
    elif type == "code":
        pattern = r"```(?:problog|[a-z]*)?\n(.*?)```"
    else:
        raise ValueError("you must choose from ['report','code','other']")
    
    matches = re.findall(pattern, text, re.DOTALL)
    
    if not matches:
        pattern = r"```(?:json|[a-z]*)?\n(.*?)```"
        matches = re.findall(pattern, text, re.DOTALL)
    
    for match in matches:
        match = match.strip()
        
        try:
            # 直接尝试解析JSON
            match_json = json.loads(match)
        except json.JSONDecodeError:
            # 智能处理转义序列
            try:
                # 创建一个映射来临时替换特殊序列
                replacements = {
                    r'\\\\=': '___DOUBLE_BACKSLASH_EQUALS___',
                    r'\\=': '___BACKSLASH_EQUALS___',
                    r'\\n': '___BACKSLASH_N___',
                    '\n': '\\n',  # 将实际的换行符替换为转义的换行符
                }
                
                processed_match = match
                
                # 首先处理实际的换行符
                processed_match = processed_match.replace('\n', '\\n')
                
                # 然后处理其他特殊序列（按长度降序，确保先处理长的）
                for pattern, replacement in sorted(replacements.items(), key=lambda x: len(x[0]), reverse=True):
                    if pattern != '\n':  # 跳过已处理的换行符
                        processed_match = processed_match.replace(pattern, replacement)
                
                # 尝试解析JSON
                match_json = json.loads(processed_match)
                
                # 还原特殊序列
                if isinstance(match_json, dict) and "Code" in match_json:
                    code = match_json["Code"]
                    # 还原顺序也很重要
                    for pattern, replacement in replacements.items():
                        if pattern != '\n':  # 不需要还原换行符
                            code = code.replace(replacement, pattern)
                    match_json["Code"] = code
                
            except json.JSONDecodeError as e:
                print(f"JSON解析失败: {e}")
                print(f"原始内容: {repr(match)}")
                print(f"处理后内容: {repr(processed_match)}")
                continue
        
        # 根据类型处理解析结果
        if type == "other":
            blocks.append({ext_mark: match_json})
        elif type == "code":
            if isinstance(match_json, dict) and "HASH" in match_json and "Code" in match_json:
                blocks.append({match_json["HASH"]: match_json["Code"]})
            else:
                blocks.append({"unknown": f"code: invalid structure - {match_json}"})
        elif type == "report":
            if isinstance(match_json, dict) and "HASH" in match_json:
                blocks.append({match_json["HASH"]: match_json})
            else:
                blocks.append({"unknown": f"report: invalid structure - {match_json}"})
    
    return blocks

In [7]:

model = 'Here is the completed code segment for the {LANGDA} placeholder:\n\n```problog\n{"HASH": "CD6DE9CB","Code": "query_pop([C1,D1,C2,D2]) :- \n    density(C1,D1),\n    density(C2,D2),\n    C1 \\\\= C2,\n    abs(D1 - D2) =< max(D1,D2)*0.05."}\n```\n\nThis implementation:\n1. Uses the `density/2` predicate to get population densities for two countries\n2. Ensures they are different countries with `C1 \\= C2`\n3. Checks if their densities are within 5% of each other using arithmetic comparison\n4. Returns the country names and densities in the format [C1,D1,C2,D2]\n\nThe 5% threshold can be adjusted by changing the 0.05 factor as needed.'

rep = _find_all_blocks("code", model)
print(rep)

[{'CD6DE9CB': 'query_pop([C1,D1,C2,D2]) :- \n    density(C1,D1),\n    density(C2,D2),\n    C1 \\\\= C2,\n    abs(D1 - D2) =< max(D1,D2)*0.05.'}]


In [19]:
def _replace_placeholder(template: str, replacement_list, placeholder="{{LANGDA}}") -> str:
    """
    Replaces placeholders in a template with items from a replacement list.
    
    args:
        template: template with placeholders
        replacement_list: the list of content that will fit into the placeholder.
        placeholder: default as {{LANGDA}}
        - if the value is None, the corresponding placeholder remains unchanged. 
        - valid input forms: List[str] or List[dict]
    """
    # Extract values from replacement items
    replace_str_list = []
    if replacement_list and all(isinstance(item, dict) for item in replacement_list):
        for item in replacement_list:
            _, value = next(iter(item.items()))
            replace_str_list.append(value)
    else:
        replace_str_list = replacement_list

    # Split the template by placeholder
    segments = template.split(placeholder)
    result = segments[0]
    
    # Process each segment after the first
    for i, seg in enumerate(segments[1:]):
        # Check if we have a replacement for this placeholder
        if i < len(replace_str_list) and replace_str_list[i] is not None:
            replace_text = replace_str_list[i]
            
            # Check for duplicate punctuation at the boundaries
            if replace_text and seg:
                replace_ends_with_punct = replace_text.rstrip()[-1:] in ".,;" if replace_text.rstrip() else False
                seg_starts_with_punct = seg.lstrip()[:1] in ".,;" if seg.lstrip() else False
                
                # Handle duplicate punctuation
                if replace_ends_with_punct and seg_starts_with_punct:
                    # Find position where the actual text ends in replace_text
                    replace_text_end = len(replace_text.rstrip())
                    # Find position where the actual text starts in seg
                    seg_text_start = len(seg) - len(seg.lstrip())
                    
                    # Add replace_text without the trailing punctuation
                    result += replace_text[:replace_text_end-1] 
                    # Add seg with its leading whitespace and punctuation
                    result += seg[:seg_text_start+1] + seg[seg_text_start+1:]
                    continue
            
            # Normal case: just add the replacement and segment
            result += replace_text
        else:
            # No replacement available, keep the placeholder
            result += placeholder
        
        result += seg
        
    return result

In [20]:
model = """
 
digit(O), all_different([O,R,N,Y,E,D]),
 sumdigit(C2, E, O, N, C3),
 
 digit(M), all_different([M,O,R,N,Y,E,D]),
 sumdigit(C3, S, M, O, C4),
 
{{LANGDA}}
  . 
 all_different([S,E,N,D,M,O,R,Y]).
sumdigit(C, A, B, S, 0) :-
 X is C + A + B,
 X < 10,
 S = X.
sumdigit(C, A, B, S, 1) :-
 X is C + A + B,
 X >= 10,
 S is X - 10.
"""
match = """digit(S), leftdigit(S), all_different([S,M,O,R,N,Y,E,D]),
 sumdigit(C4, 0, 0, M, _).  """

print(_replace_placeholder(model,[match]))




digit(O), all_different([O,R,N,Y,E,D]),
 sumdigit(C2, E, O, N, C3),

 digit(M), all_different([M,O,R,N,Y,E,D]),
 sumdigit(C3, S, M, O, C4),

digit(S), leftdigit(S), all_different([S,M,O,R,N,Y,E,D]),
 sumdigit(C4, 0, 0, M, _)
  . 
 all_different([S,E,N,D,M,O,R,Y]).
sumdigit(C, A, B, S, 0) :-
 X is C + A + B,
 X < 10,
 S = X.
sumdigit(C, A, B, S, 1) :-
 X is C + A + B,
 X >= 10,
 S is X - 10.



In [ ]:
model = """
//the completed original code here
```
</Final_Answer>

*** split ***
In section <origin_code> and <generated_code> you will be give two codes,
- in <origin_code> there's incomplete code with {{LANGDA}} blocks.
- in <generated_
"""


lines = model.split("*** split ***")
print(lines[0])
print("======================")
print(lines[1])



//the completed original code here
```
</Final_Answer>



In section <origin_code> and <generated_code> you will be give two codes,
- in <origin_code> there's incomplete code with <langda> blocks.
- in <generated_



In [14]:
def merge_with_overlap_tokens(s1: str, s2: str) -> str:
    tokens1 = s1.split()
    tokens2 = s2.split()
    max_overlap = min(len(tokens1), len(tokens2))
    for k in range(max_overlap, 0, -1):
        if tokens1[-k:] == tokens2[:k]:
            return " ".join(tokens1 + tokens2[k:])
    return " ".join(tokens1 + tokens2)

# 测试
a = """This new 
cat 
 
"""

b = """  new   cat is cute,
but I dont like it
"""
print( merge_with_overlap_tokens(a, b) )
# 输出: This new cat is cute


This new cat is cute, but I dont like it


In [30]:
model = """
% -------------------------
% Basic rules of rock-paper-scissors
% -------------------------
% Three gestures
move(rock).
move(paper).
move(scissor).
% Win-lose relationship: X beats Y
beats(rock, scissor).
beats(scissor, paper).
beats(paper, rock).
% -------------------------
% Calculate the result of the game
% -------------------------
result(X, X, draw) :-

{{LANGDA}}
.
result(X, Y, win) :-
{{LANGDA}}
.
result(X, Y, lose) :-
{{LANGDA}}
.
% End of recursion: empty list corresponds to empty result
play([], [], []).
% Recursive advancement: take out each round of gestures, calculate the results, and continue
play([P1|P1T], [P2|P2T], [R|Rs]) :-
% The correct call is result(P1,P2,R), not semicolon
result(P1, P2, R),
% (Optional) Update the score according to R
play(P1T, P2T, Rs).
compute_score([], 0).
{{LANGDA}}
.
{{LANGDA}}
.
{{LANGDA}}
.
compute_score([draw | Rs], S) :- compute_score(Rs, S1), S is S1.
determine_winner(P1Moves,P2Moves,Winner) :- 
 
{{LANGDA}}
,
compute_score(Results,S), 
( S > 0, Winner = player1 
; S < 0, Winner = player2 
; S = 0, Winner = draw 
).
query(determine_winner([rock,rock,rock],[paper,paper,scissor],W)).
"""

In [36]:
import re
import json
import uuid
import hashlib
from problog.program import PrologString, Clause, AnnotatedDisjunction, Term
from typing import Literal, List, Union, Tuple, Dict, Any
from langgraph.graph import END, StateGraph
from state import BasicState, Mode
from config import paths

codes = [
{"3F277A35": "move(X)."},{"D91BB7A0": "beats(X, Y)."},{"0940BB04": "beats(Y, X)."},
{"583E41B6": "compute_score([win | Rs], S) :- compute_score(Rs, S1), S is S1 + 1."},
{"D324979D": "compute_score([lose | Rs], S) :- compute_score(Rs, S1), S is S1 - 1."},
{"FD850DEC": "compute_score([draw | Rs], S) :- compute_score(Rs, S1), S is S1."},
{"3FEB17D7": "play(P1Moves, P2Moves, Results),"}]

import re

def _tokenize_problog(s):
    # Tokenize Prolog code into meaningful units, including punctuation
    pattern = r":-|\w+|[^\w\s]"
    return [(m.group(), m.start(), m.end()) for m in re.finditer(pattern, s)]

def _merge_problog_preserve(s1, s2):
    # Tokenize both strings
    tokens1 = _tokenize_problog(s1)
    tokens2 = _tokenize_problog(s2)

    texts1 = [t for t, _, _ in tokens1]
    texts2 = [t for t, _, _ in tokens2]
    
    # Find the longest token overlap
    max_k = min(len(texts1), len(texts2))
    for k in range(max_k, 0, -1):
        if texts1[-k:] == texts2[:k]:
            # Calculate where to splice in the original s2
            j_start = tokens2[k-1][2]
            return s1 + s2[j_start:]
    
    # Fallback: no overlap
    return s1 + s2

def _replace_placeholder(template:str, replacement_list:Union[List[str],List[dict]], placeholder="{{LANGDA}}") -> str:
    """
    Replaces placeholders in a template with items from a replacement list.
    
    args:
        template: template with placeholders
        replacement_list: the list of content that will fit into the placeholder.
        placeholder: default as {{LANGDA}}
        - if the value is None, the corresponding placeholder remains unchanged. 
        - valid input forms: List[str] or List[dict]
    """

    # Extract values from replacement items
    replace_str_list = []
    if replacement_list and all(isinstance(item, dict) for item in replacement_list):
        for item in replacement_list:
            _, value = next(iter(item.items()))
            replace_str_list.append(value)
    else:
        replace_str_list = replacement_list

    # Split the template by placeholder
    segments = template.split(placeholder)
    result = segments[0]
    
    # Process each segment after the first
    for i, seg in enumerate(segments[1:]):

        replace_text = replace_str_list[i]
        # Check if we have a replacement for this placeholder
        if i < len(replace_str_list) and replace_text is not None:
            # !!! SYNTAX FIX !!!
            # deal with overlap: segment[0] & "overlap text" + "overlap text" & replace_text
            result = _merge_problog_preserve(result, replace_text)
        else:
            # No replacement available, keep the placeholder
            result += placeholder

        # !!! SYNTAX FIX !!!
        # deal with overlap: replace_text & "overlap text" + "overlap text" & segment[1]
        result = _merge_problog_preserve(result, seg)
        
    return result

print(_replace_placeholder(model,codes))


% -------------------------
% Basic rules of rock-paper-scissors
% -------------------------
% Three gestures
move(rock).
move(paper).
move(scissor).
% Win-lose relationship: X beats Y
beats(rock, scissor).
beats(scissor, paper).
beats(paper, rock).
% -------------------------
% Calculate the result of the game
% -------------------------
result(X, X, draw) :-

move(X).
result(X, Y, win) :-
beats(X, Y).
result(X, Y, lose) :-
beats(Y, X).
% End of recursion: empty list corresponds to empty result
play([], [], []).
% Recursive advancement: take out each round of gestures, calculate the results, and continue
play([P1|P1T], [P2|P2T], [R|Rs]) :-
% The correct call is result(P1,P2,R), not semicolon
result(P1, P2, R),
% (Optional) Update the score according to R
play(P1T, P2T, Rs).
compute_score([], 0).
compute_score([win | Rs], S) :- compute_score(Rs, S1), S is S1 + 1.
compute_score([lose | Rs], S) :- compute_score(Rs, S1), S is S1 - 1.
compute_score([draw | Rs], S) :- compute_score(Rs, S1)

In [ ]:
import time
import re


def tokenize_problog(s):
    pattern = r":-|\w+|[(),.]"
    return [(m.group(), m.start()) for m in re.finditer(pattern, s)]

def merge_problog_block(s1, s2):
    # Tokenize both strings
    tokens1 = tokenize_problog(s1)
    tokens2 = tokenize_problog(s2)
    t1 = [t for t, _ in tokens1]
    t2 = [t for t, _ in tokens2]
    max_k = min(len(t1), len(t2))
    for k in range(0, max_k, 1):
        if t1[-k:] == t2[:k]:
            j_start = tokens2[k][1] if k < len(tokens2) else len(s2)
            return s1 + s2[j_start:]
    return s1 + s2

a = """digit(img_1,1).
digit(img_0,0).
addition(X,Y,Z) :- 
"""

b = """ 
 addition(X, Y, Z) :-  digit(X, A), digit(Y, B), Z is A + B.
"""
c = """
.

query(addition(img_5,img_6,Z)).

"""
# # 调整这里的 num_lines 和 overlap_lines，看看不同规模下的速度
# a, b = generate_long_strings(num_lines=50000, overlap_lines=2000)

# 测速
start = time.perf_counter()
result = merge_problog_block(a, b)
result = merge_problog_block(result, c)
end = time.perf_counter()

print(f"合并后总长度：{result} 字符")
print(f"运行时间：{end - start:.4f} 秒")


['digit', '(', 'img_1', ',', '1', ')', '.', 'digit', '(', 'img_0', ',', '0', ')', '.', 'addition', '(', 'X', ',', 'Y', ',', 'Z', ')', ':-']
['addition', '(', 'X', ',', 'Y', ',', 'Z', ')', ':-', 'digit', '(', 'X', ',', 'A', ')', ',', 'digit', '(', 'Y', ',', 'B', ')', ',', 'Z', 'is', 'A', 'B', '.']
['digit', '(', 'img_1', ',', '1', ')', '.', 'digit', '(', 'img_0', ',', '0', ')', '.', 'addition', '(', 'X', ',', 'Y', ',', 'Z', ')', ':-'] || []
[':-'] || ['addition']
[')', ':-'] || ['addition', '(']
['Z', ')', ':-'] || ['addition', '(', 'X']
[',', 'Z', ')', ':-'] || ['addition', '(', 'X', ',']
['Y', ',', 'Z', ')', ':-'] || ['addition', '(', 'X', ',', 'Y']
[',', 'Y', ',', 'Z', ')', ':-'] || ['addition', '(', 'X', ',', 'Y', ',']
['X', ',', 'Y', ',', 'Z', ')', ':-'] || ['addition', '(', 'X', ',', 'Y', ',', 'Z']
['(', 'X', ',', 'Y', ',', 'Z', ')', ':-'] || ['addition', '(', 'X', ',', 'Y', ',', 'Z', ')']
['addition', '(', 'X', ',', 'Y', ',', 'Z', ')', ':-'] || ['addition', '(', 'X', ',', 'Y', ',